# CenterNet Heatmap 改进消融实验

三个改进方向：
1. **各向异性 Gaussian 核**
2. **自适应 Sigma 策略**
3. **Focal Loss 超参数消融**

在 Mini-COCO（4000 train / 1000 val）上运行，输出各组 AP 对比。

## 1. 环境准备

In [ ]:
import os, sys

PROJECT_DIR = '/kaggle/working/pytorch_simple_CenterNet_45'
DATA_DIR    = '/kaggle/working/data'
COCO_DIR    = os.path.join(DATA_DIR, 'coco')
IMAGES_DIR  = os.path.join(COCO_DIR, 'images')
ANN_DIR     = os.path.join(COCO_DIR, 'annotations')

# 最早阶段写入 sys.path，后续所有 import 都能找到项目模块
for p in [PROJECT_DIR, os.path.join(PROJECT_DIR, 'lib')]:
    if p not in sys.path:
        sys.path.insert(0, p)

for d in [DATA_DIR, COCO_DIR, IMAGES_DIR, ANN_DIR]:
    os.makedirs(d, exist_ok=True)

print('PROJECT_DIR:', PROJECT_DIR)
print('sys.path[0:2]:', sys.path[0:2])

In [ ]:
!pip install pycocotools easydict progress -q

## 2. 克隆代码库

In [ ]:
import os

if not os.path.exists(PROJECT_DIR):
    !git clone https://github.com/oncetrange/pytorch_simple_CenterNet_45 {PROJECT_DIR}
else:
    print('仓库已存在，跳过克隆')

os.chdir(PROJECT_DIR)
print('工作目录:', os.getcwd())

## 3. 数据准备

### 3.1 下载 COCO val2017

In [ ]:
import zipfile, os

RAW_DIR = '/kaggle/working/raw'
os.makedirs(RAW_DIR, exist_ok=True)

def is_valid_zip(path):
    try:
        with zipfile.ZipFile(path, 'r') as z:
            return z.testzip() is None
    except Exception:
        return False

def download_and_extract(url, zip_path, extract_dir, check_file):
    if os.path.exists(check_file):
        print(f'已存在，跳过: {os.path.basename(check_file)}')
        return
    # 若 zip 损坏或不完整则删除重下
    if os.path.exists(zip_path) and not is_valid_zip(zip_path):
        print(f'zip 损坏，删除重下: {os.path.basename(zip_path)}')
        os.remove(zip_path)
    if not os.path.exists(zip_path):
        print(f'下载 {os.path.basename(zip_path)} ...')
        ret = os.system(f'wget --show-progress -O {zip_path} {url}')
        if ret != 0 or not is_valid_zip(zip_path):
            if os.path.exists(zip_path): os.remove(zip_path)
            raise RuntimeError(f'下载失败: {url}')
    print(f'解压 {os.path.basename(zip_path)} ...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_dir)
    if not os.path.exists(check_file):
        raise FileNotFoundError(f'解压后仍未找到: {check_file}')
    print(f'完成: {os.path.basename(check_file)}')

download_and_extract(
    url='http://images.cocodataset.org/zips/val2017.zip',
    zip_path=os.path.join(RAW_DIR, 'val2017.zip'),
    extract_dir=RAW_DIR,
    check_file=os.path.join(RAW_DIR, 'val2017'),
)

download_and_extract(
    url='http://images.cocodataset.org/annotations/annotations_trainval2017.zip',
    zip_path=os.path.join(RAW_DIR, 'annotations_trainval2017.zip'),
    extract_dir=RAW_DIR,
    check_file=os.path.join(RAW_DIR, 'annotations', 'instances_val2017.json'),
)

print('数据准备完成')

### 3.2 切分数据集

In [ ]:
import json, random, shutil
from tqdm import tqdm

ANN_SRC   = os.path.join(RAW_DIR, 'annotations', 'instances_val2017.json')
IMG_SRC   = os.path.join(RAW_DIR, 'val2017')
TRAIN_ANN = os.path.join(ANN_DIR, 'instances_train2017.json')
VAL_ANN   = os.path.join(ANN_DIR, 'instances_val2017.json')

if not os.path.exists(TRAIN_ANN):
    with open(ANN_SRC) as f:
        data = json.load(f)
    imgs = data['images']
    random.seed(42)
    random.shuffle(imgs)
    train_imgs, val_imgs = imgs[:4000], imgs[4000:5000]
    train_ids = {i['id'] for i in train_imgs}
    val_ids   = {i['id'] for i in val_imgs}
    base = {k: data[k] for k in ('info', 'licenses', 'categories')}
    for split_imgs, split_ids, path in [
        (train_imgs, train_ids, TRAIN_ANN),
        (val_imgs,   val_ids,   VAL_ANN)
    ]:
        d = {**base, 'images': split_imgs,
             'annotations': [a for a in data['annotations'] if a['image_id'] in split_ids]}
        with open(path, 'w') as f:
            json.dump(d, f)
    print(f'切分完成: train={len(train_imgs)}, val={len(val_imgs)}')
else:
    print('标注文件已存在，跳过切分')

# 复制图片（断点续传）
to_copy = [f for f in os.listdir(IMG_SRC)
           if not os.path.exists(os.path.join(IMAGES_DIR, f))]
if to_copy:
    print(f'复制 {len(to_copy)} 张图片...')
    for f in tqdm(to_copy):
        shutil.copy(os.path.join(IMG_SRC, f), os.path.join(IMAGES_DIR, f))
else:
    print('图片已就绪')

In [ ]:
# 软链接：train2017 / val2017 -> images
for split in ('train2017', 'val2017'):
    link = os.path.join(COCO_DIR, split)
    if os.path.islink(link): os.unlink(link)
    if not os.path.exists(link):
        os.symlink(IMAGES_DIR, link)
print('软链接就绪')

### 3.3 数据清洗

In [ ]:
import cv2

def clean_annotations(ann_path, img_dir):
    with open(ann_path) as f:
        data = json.load(f)
    valid_imgs, valid_ids, bad = [], set(), 0
    for img in tqdm(data['images'], desc=os.path.basename(ann_path)):
        p = os.path.join(img_dir, img['file_name'])
        if os.path.exists(p) and os.path.getsize(p) > 0 and cv2.imread(p) is not None:
            valid_imgs.append(img)
            valid_ids.add(img['id'])
        else:
            bad += 1
            if os.path.exists(p): os.remove(p)
    data['images']      = valid_imgs
    data['annotations'] = [a for a in data['annotations'] if a['image_id'] in valid_ids]
    with open(ann_path, 'w') as f:
        json.dump(data, f)
    print(f'  保留 {len(valid_imgs)} 张，剔除 {bad} 张')

clean_annotations(TRAIN_ANN, IMAGES_DIR)
clean_annotations(VAL_ANN,   IMAGES_DIR)

## 4. 实验配置

In [ ]:
import sys

PYTHON = sys.executable
TRAIN_SCRIPT = os.path.join(PROJECT_DIR, 'train.py')

COMMON = (
    f'--root_dir {PROJECT_DIR} '
    f'--data_dir {DATA_DIR} '
    '--dataset coco '
    '--arch resdcn_18 '
    '--img_size 512 '
    '--batch_size 32 '
    '--lr 1.25e-4 '
    '--num_epochs 30 '
    '--lr_step 90,120 '
    '--num_workers 4'
)

EXPERIMENTS = [
    {'name': 'baseline',     'desc': '原始实现（各向同性 + 固定 sigma + CornerNet focal）', 'extra': ''},
    {'name': 'aniso_gauss',  'desc': '各向异性 Gaussian（按宽高比调整 x/y 半径）',          'extra': '--gaussian_type anisotropic'},
    {'name': 'sigma_adapt',  'desc': '自适应 sigma（小目标 4，中目标 6，大目标 8）',          'extra': '--sigma_mode adaptive'},
    {'name': 'focal_gamma2', 'desc': 'Focal γ=2（减弱 Gaussian 覆盖区负样本抑制）',         'extra': '--focal_gamma 2.0'},
    {'name': 'focal_beta4',  'desc': 'Focal β=4（加强对高置信度假正样本惩罚）',              'extra': '--focal_beta 4.0'},
    {'name': 'aniso_adapt',  'desc': '组合：各向异性 + 自适应 sigma',                       'extra': '--gaussian_type anisotropic --sigma_mode adaptive'},
]

print(f'共 {len(EXPERIMENTS)} 组实验')
for e in EXPERIMENTS:
    print(f"  [{e['name']:15s}] {e['desc']}")

## 5. 训练

> Kaggle T4：resdcn_18 每轮约 40–60s，30 轮约 25–30 分钟/实验，6 组共约 3 小时。  
> 如时间有限，可将 `--num_epochs` 改为 `10` 快速对比趋势。

In [ ]:
import subprocess

# 把 PROJECT_DIR 注入子进程的 PYTHONPATH，让 train.py 能找到 datasets/nets/utils
train_env = os.environ.copy()
train_env['PYTHONPATH'] = PROJECT_DIR + ':' + os.path.join(PROJECT_DIR, 'lib') \
                          + ':' + train_env.get('PYTHONPATH', '')

def run_training(exp):
    cmd = f"{PYTHON} {TRAIN_SCRIPT} {COMMON} --log_name {exp['name']} {exp['extra']}"
    print(f"\n{'='*60}\n实验: {exp['name']}  |  {exp['desc']}\n{'='*60}")
    result = subprocess.run(cmd, shell=True, cwd=PROJECT_DIR, env=train_env)
    status = '完成' if result.returncode == 0 else f'异常退出 code={result.returncode}'
    print(f'[{status}] {exp["name"]}')

for exp in EXPERIMENTS:
    run_training(exp)

## 6. 评估

In [ ]:
import importlib, time, torch, numpy as np
from tqdm import tqdm

import utils.image, utils.losses, datasets.coco
for mod in [utils.image, utils.losses, datasets.coco]:
    importlib.reload(mod)

from datasets.coco import COCO_eval
from nets.resdcn import get_pose_net
from utils.utils import load_model
from utils.image import transform_preds
from utils.post_process import ctdet_decode

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FPS_WARMUP = 50  # 前 N 张仅做 GPU 预热，不计入 FPS
print('推理设备:', DEVICE)

def evaluate_experiment(exp_name):
    ckpt = os.path.join(PROJECT_DIR, 'ckpt', exp_name, 'checkpoint.t7')
    if not os.path.exists(ckpt):
        print(f'[跳过] 未找到: {ckpt}')
        return None

    dataset = COCO_eval(DATA_DIR, 'val', test_scales=[1.], test_flip=False)
    loader  = torch.utils.data.DataLoader(
        dataset, batch_size=1, shuffle=False,
        num_workers=2, pin_memory=True, collate_fn=dataset.collate_fn)

    model = get_pose_net(num_layers=18, num_classes=dataset.num_classes)
    model = load_model(model, ckpt).to(DEVICE)
    model.eval()

    results = {}
    forward_times = []  # 仅记录 model forward 耗时（秒）

    with torch.no_grad():
        for idx, inputs in enumerate(tqdm(loader, desc=exp_name, leave=False)):
            img_id, inputs = inputs[0]
            dets_all = []
            for scale in inputs:
                inputs[scale]['image'] = inputs[scale]['image'].to(DEVICE)

                # FPS 计时：仅对 model forward 计时，使用 cuda.synchronize 保证精度
                if DEVICE.type == 'cuda':
                    torch.cuda.synchronize()
                t0 = time.perf_counter()
                output = model(inputs[scale]['image'])[-1]
                if DEVICE.type == 'cuda':
                    torch.cuda.synchronize()
                t1 = time.perf_counter()

                if idx >= FPS_WARMUP:  # 跳过前 N 张 warmup
                    forward_times.append(t1 - t0)

                dets = ctdet_decode(*output, K=100).detach().cpu().numpy().reshape(-1, 6)
                top_preds = {}
                dets[:, :2] = transform_preds(dets[:, :2], inputs[scale]['center'],
                                              inputs[scale]['scale'],
                                              (inputs[scale]['fmap_w'], inputs[scale]['fmap_h']))
                dets[:, 2:4] = transform_preds(dets[:, 2:4], inputs[scale]['center'],
                                               inputs[scale]['scale'],
                                               (inputs[scale]['fmap_w'], inputs[scale]['fmap_h']))
                for j in range(dataset.num_classes):
                    inds = dets[:, -1] == j
                    top_preds[j + 1] = (dets[inds, :5] / scale).astype(np.float32)
                dets_all.append(top_preds)

            bbox_scores = {j: np.concatenate([d[j] for d in dets_all], axis=0)
                           for j in range(1, dataset.num_classes + 1)}
            scores = np.hstack([bbox_scores[j][:, 4] for j in range(1, dataset.num_classes + 1)])
            if len(scores) > 100:
                thresh = np.partition(scores, len(scores) - 100)[len(scores) - 100]
                for j in range(1, dataset.num_classes + 1):
                    bbox_scores[j] = bbox_scores[j][bbox_scores[j][:, 4] >= thresh]
            results[img_id] = bbox_scores

    fps = 1.0 / (sum(forward_times) / len(forward_times)) if forward_times else 0.0
    print(f'  FPS (model forward only, warmup={FPS_WARMUP}): {fps:.1f}')

    save_dir = os.path.join(PROJECT_DIR, 'ckpt', exp_name)
    stats = dataset.run_eval(results, save_dir=save_dir)
    return stats, fps


In [ ]:
all_results = {}

for exp in EXPERIMENTS:
    print(f"\n评估: {exp['name']}")
    ret = evaluate_experiment(exp['name'])
    if ret is not None:
        stats, fps = ret
        all_results[exp['name']] = {
            'desc': exp['desc'],
            'AP': stats[0], 'AP50': stats[1], 'AP75': stats[2],
            'APs': stats[3], 'APm': stats[4], 'APl': stats[5],
            'FPS': round(fps, 1),
        }
        print(f"  AP={stats[0]:.3f}  AP50={stats[1]:.3f}  APs={stats[3]:.3f}  "
              f"APm={stats[4]:.3f}  APl={stats[5]:.3f}  FPS={fps:.1f}")


## 7. 结果汇总

In [ ]:
import pandas as pd

if all_results:
    baseline_ap = all_results.get('baseline', {}).get('AP', 0)
    rows = []
    for name, r in all_results.items():
        rows.append({
            '实验': name, '描述': r['desc'],
            'AP': round(r['AP'], 3), 'AP50': round(r['AP50'], 3),
            'AP75': round(r['AP75'], 3), 'APs': round(r['APs'], 3),
            'APm': round(r['APm'], 3), 'APl': round(r['APl'], 3),
            'FPS': r.get('FPS', '--'),
            'ΔAP': round(r['AP'] - baseline_ap, 3) if name != 'baseline' else '--',
        })
    df = pd.DataFrame(rows).set_index('实验')
    pd.set_option('display.max_colwidth', 55)
    display(df)
else:
    print('暂无结果')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if all_results:
    names = list(all_results.keys())
    metrics = ['AP', 'AP50', 'APs', 'APm', 'APl']
    x = np.arange(len(names))
    w = 0.15

    fig, ax = plt.subplots(figsize=(14, 5))
    for i, m in enumerate(metrics):
        ax.bar(x + i * w, [all_results[n][m] for n in names], w, label=m)
    ax.set_xticks(x + w * 2)
    ax.set_xticklabels(names, rotation=15, ha='right')
    ax.set_ylabel('Score')
    ax.set_title('CenterNet Heatmap Ablation (Mini-COCO, ResNet-18, 30 epochs)')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    out_path = '/kaggle/working/ablation_results.png'
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f'图表已保存: {out_path}')